# Camadas 1 e 2 com o dicionário final

Calcula os principais resultados das Camadas 1 e 2 com a versão final dos dicionários, na qual termos de sentido genérico foram retirados após a auditoria lexical (Apêndice A).

**Termos não utilizados na versão final**

- Camada 1: `meio` (Meio ambiente), `reforma` (Agricultura), `porto` (Infraestrutura), `federal` e `estado` (Federalismo), `presidente` (Democracia).
- Camada 2: `pessoa` e `pessoas` (Raça, indígenas e minorias), `vida` e `familia` (Religião e moral).

Os notebooks 17 a 22 utilizam a versão completa dos dicionários, mantida para fins de comparação.

**Resultados gerados:** Tabela 5.3 (R1, R2, R3), Figura 5.2, Tabela 5.4 e Figura 5.3, Tabela 5.5, Tabela 5.14 (controle do tamanho), Tabela 5.15 (regras e janelas), Tabela 5.16 (versões do dicionário), razões de chances da Camada 2 e Tabela A.1. Os arquivos correspondem a `results/tables/dicionario_final/`.

**Dados de entrada:** corpus pré-processado em `legislatura/data/artigo1_preprocessado`, gerado por `scripts/01_preprocess_corpus.py` (ver `data/README_DATA.md`).

**Execução:** Google Colab → *Ambiente de execução → Executar tudo*. Tempo estimado: 20 a 30 minutos.


In [ ]:
# 1) Conectar o Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Localizar a pasta pré-processada da dissertação
from pathlib import Path
RAIZ = Path('/content/drive/MyDrive')
candidatos = [RAIZ / 'legislatura' / 'data' / 'artigo1_preprocessado']
candidatos += list(RAIZ.glob('*/legislatura/data/artigo1_preprocessado'))
candidatos += list(RAIZ.glob('*/data/artigo1_preprocessado'))
PRE = next((c for c in candidatos if (c / 'corpus_artigo1_metadata.csv').exists()), None)
if PRE is None:
    raise SystemExit('Pasta artigo1_preprocessado não encontrada no Drive montado (ver Dados de entrada no topo).')
print('Pasta encontrada:', PRE)

SAIDA = RAIZ / 'MESTRADO' / 'resultados_dicionario_final'
SAIDA.mkdir(parents=True, exist_ok=True)
PERCENTIL_R3 = 0.50   # limiar fixo da R3 = mediana global das densidades entre os pares sinalizados por R1

In [ ]:
# 3) Ler os discursos pré-processados (mesma leitura dos notebooks 17 a 22)
import pandas as pd
import numpy as np
from collections import Counter
LEGS = [52, 53, 54, 55, 56, 57]
meta = pd.read_csv(PRE / 'corpus_artigo1_metadata.csv', usecols=['id_legislatura'])
textos, legs = [], []
for leg in LEGS:
    linhas = (PRE / 'by_legislature' / f'leg{leg}.txt').read_text(encoding='utf-8').splitlines()
    n_meta = int((meta['id_legislatura'] == leg).sum())
    if n_meta != len(linhas):
        raise ValueError(f'Leg {leg}: metadados={n_meta} texto={len(linhas)}')
    textos += linhas; legs += [leg] * len(linhas)
    print(f'Legislatura {leg}: {len(linhas):,} discursos')
df = pd.DataFrame({'_leg': legs, '_tokens': [t.split() for t in textos]})
del textos, meta
df['_n'] = df['_tokens'].map(len)
N = len(df)
LEG = df['_leg'].values
print(f'TOTAL: {N:,} discursos  (dissertação: 439.627)')

In [ ]:
# 4) Dicionários: versão completa (notebooks 17 a 22) e versão final, sem os termos genéricos
THEMES_ORIG = {
 "Educação": {"educacao","escola","escolas","ensino","professor","professores","aluno","alunos","universidade","universidades","creche","alfabetizacao","fundeb","enem","estudante","estudantes","pesquisa","ciencia","cientifico","tecnologia"},
 "Saúde": {"saude","sus","hospital","hospitais","medico","medicos","enfermeiro","enfermeiros","enfermeiras","vacina","vacinas","pandemia","covid","doenca","doencas","tratamento","medicamento","medicamentos","pacientes","atendimento"},
 "Economia, trabalho e renda": {"economia","economico","economica","emprego","empregos","trabalho","trabalhador","trabalhadores","renda","salario","salarios","inflacao","juros","credito","industria","empresas","empresa","mercado","crescimento","desenvolvimento","investimento","investimentos","tributaria","imposto","impostos"},
 "Segurança pública e violência": {"seguranca","violencia","crime","crimes","criminalidade","policia","policial","policiais","prisao","presidio","armas","arma","trafico","homicidio","homicidios","feminicidio","militar","militares"},
 "Meio ambiente e clima": {"meio","ambiente","ambiental","amazonia","floresta","florestas","desmatamento","clima","climatica","climatico","sustentavel","sustentabilidade","biodiversidade","indigena","indigenas","garimpo","queimadas","agua","saneamento"},
 "Infraestrutura, transporte e habitação": {"infraestrutura","transporte","transportes","rodovia","rodovias","estrada","estradas","ferrovia","ferrovias","porto","portos","aeroporto","aeroportos","energia","eletrica","habitacao","moradia","saneamento","internet","telecomunicacoes","obra","obras"},
 "Proteção social e desigualdade": {"pobreza","fome","desigualdade","desigualdades","assistencia","social","bolsa","familia","beneficio","beneficios","previdencia","aposentadoria","aposentados","vulnerabilidade","moradia","direitos","mulheres","criancas","juventude","idosos","deficiencia"},
 "Federalismo e municípios": {"municipio","municipios","prefeito","prefeitos","prefeitura","prefeituras","estado","estados","federacao","federal","governador","governadores","regional","regionais","regiao","regioes","nordeste","norte","sul","sudeste","centro"},
 "Democracia e instituições": {"democracia","democratico","democratica","constituicao","constitucional","congresso","camara","senado","parlamento","governo","presidente","presidencia","republica","eleicao","eleicoes","voto","impeachment","stf","supremo","judiciario","justica"},
 "Direitos e cidadania": {"direito","direitos","cidadania","cidadao","cidadaos","igualdade","liberdade","justica","mulher","mulheres","negros","racismo","discriminacao","lgbt","indigena","indigenas","quilombolas","criancas","adolescentes"},
 "Agricultura e alimentação": {"agricultura","agricola","agropecuaria","agronegocio","produtor","produtores","rural","campo","alimentos","alimento","comida","fome","pecuaria","safra","assentamento","reforma","agraria"},
}
SETORIZADO_ORIG = {
 "agricultura_rural": {"agricultura","agricola","agropecuaria","agronegocio","produtor","produtores","rural","campo","pecuaria","safra","assentamento"},
 "empresas_industria": {"empresa","empresas","industria","industrial","comercio","comerciante","empreendedor","empreendedores","microempresa","empresarios"},
 "trabalhadores_sindicatos": {"trabalhador","trabalhadores","sindicato","sindicatos","servidor","servidores","categoria","categorias","funcionario","funcionarios"},
 "servidores_profissoes": {"professor","professores","medico","medicos","enfermeiro","enfermeiros","policial","policiais","militar","militares"},
 "mulheres_familia_infancia": {"mulher","mulheres","familia","familias","crianca","criancas","adolescente","adolescentes","juventude","idosos","idoso"},
 "raca_indigenas_minorias": {"negros","negro","racismo","indigena","indigenas","quilombolas","quilombola","lgbt","deficiencia","pessoa","pessoas"},
 "religiao_moral": {"religiao","religioso","religiosos","igreja","igrejas","evangelicos","catolicos","familia","vida","aborto"},
 "territorio_regiao": {"municipio","municipios","prefeito","prefeitos","nordeste","norte","amazonia","semiarido","regional","regiao","comunidade","comunidades"},
 "forcas_seguranca": {"policia","policial","policiais","militar","militares","bombeiros","guarda","forcas","armadas"},
}
NAC_A = {
 "educacao": {"educacao","escola","ensino","professor","professores","universidade","fundeb"},
 "saude": {"saude","sus","hospital","medico","medicos","vacina","pandemia","covid"},
 "economia": {"economia","emprego","trabalho","renda","salario","inflacao","juros","credito"},
 "seguranca": {"seguranca","violencia","crime","policia","armas","trafico","homicidio"},
 "infraestrutura": {"infraestrutura","transporte","rodovia","estrada","energia","saneamento","internet"},
 "meio_ambiente": {"ambiente","ambiental","amazonia","floresta","desmatamento","clima","agua"},
 "direitos_politica_social": {"direitos","cidadania","pobreza","fome","desigualdade","assistencia","previdencia"},
 "instituicoes": {"democracia","constituicao","congresso","camara","senado","governo","voto","judiciario"},
}

# Termos genéricos não utilizados na versão final
RETIRAR_C1 = {"Meio ambiente e clima": {"meio"}, "Agricultura e alimentação": {"reforma"},
              "Infraestrutura, transporte e habitação": {"porto"},
              "Federalismo e municípios": {"federal", "estado"}, "Democracia e instituições": {"presidente"}}
RETIRAR_C2 = {"raca_indigenas_minorias": {"pessoa", "pessoas"}, "religiao_moral": {"vida", "familia"}}

THEMES = {k: v - RETIRAR_C1.get(k, set()) for k, v in THEMES_ORIG.items()}
SETORIZADO = {k: v - RETIRAR_C2.get(k, set()) for k, v in SETORIZADO_ORIG.items()}
TERMOS_SET = set().union(*SETORIZADO.values())

# Versões da Tabela 5.16 (mesma construção do notebook 19, com os dicionários limpos)
NAC_B = {k: v - TERMOS_SET for k, v in NAC_A.items()}
CAMADA1_C = {k: v - TERMOS_SET for k, v in THEMES.items()
             if k not in ("Federalismo e municípios", "Agricultura e alimentação")}
TN = {"A": set().union(*NAC_A.values()), "B": set().union(*NAC_B.values()), "C": set().union(*CAMADA1_C.values())}

for k in THEMES:
    if THEMES[k] != THEMES_ORIG[k]:
        print("Camada 1 |", k, "-> não utilizados:", sorted(THEMES_ORIG[k] - THEMES[k]))
for k in SETORIZADO:
    if SETORIZADO[k] != SETORIZADO_ORIG[k]:
        print("Camada 2 |", k, "-> não utilizados:", sorted(SETORIZADO_ORIG[k] - SETORIZADO[k]))
print("Termos nacionais da Camada 2 -> A:", len(TN["A"]), "| B:", len(TN["B"]), "| C:", len(TN["C"]))

In [ ]:
# 5) CAMADA 1 — contagem por discurso e tema (dicionário final)
TEMAS = list(THEMES)
TODOS_C1 = set().union(*THEMES.values())
dist = np.zeros((N, len(TEMAS)), dtype=np.int16)
occ = np.zeros((N, len(TEMAS)), dtype=np.int32)
docfreq = Counter()
for i, toks in enumerate(df["_tokens"]):
    c = Counter(t for t in toks if t in TODOS_C1)
    docfreq.update(c.keys())
    for j, tema in enumerate(TEMAS):
        hits = [t for t in THEMES[tema] if c[t] > 0]
        dist[i, j] = len(hits)
        occ[i, j] = sum(c[t] for t in hits)
    if i % 50_000 == 0:
        print(f"  {i:,}/{N:,}")
n_tok = df["_n"].values.astype(float)
dens = np.where(n_tok[:, None] > 0, 10_000 * occ / np.maximum(n_tok[:, None], 1), 0.0)
R1 = occ > 0
R2 = (dist >= 2) | (occ >= 3)
LIMIAR = float(np.quantile(dens[R1], PERCENTIL_R3))
R3 = R1 & (dens >= LIMIAR)
print(f"Pronto. Limiar fixo da R3: {LIMIAR:.2f} ocorrências por 10 mil tokens")

In [ ]:
# 6) Conferência: a categoria Saúde é idêntica nas duas versões do dicionário
j = TEMAS.index("Saúde")
esperado = {52: 11.6, 53: 11.2, 54: 14.1, 55: 9.0, 56: 18.3, 57: 10.6}
for L in LEGS:
    v = round(100 * R2[LEG == L, j].mean(), 1)
    print(f"Saúde R2 leg {L}: {v}  (versão completa: {esperado[L]})  {'OK' if abs(v - esperado[L]) < 0.15 else 'CONFERIR'}")

In [ ]:
# 7) TABELA 5.3 — prevalência de cada tema sob R1, R2 e R3
t53 = pd.DataFrame({"R1 (%)": 100 * R1.mean(0), "R2 (%)": 100 * R2.mean(0), "R3 (%)": 100 * R3.mean(0)},
                   index=TEMAS).round(1).sort_values("R1 (%)", ascending=False)
t53.index.name = "Categoria"
t53.to_csv(SAIDA / "tabela_5_3_R1_R2_R3.csv", encoding="utf-8-sig")
print(f"Limiar fixo da R3: {LIMIAR:.2f}")
t53

In [ ]:
# 8) FIGURA 5.2 — prevalência por legislatura sob R2 (mapa de calor)
import matplotlib.pyplot as plt
import seaborn as sns
heat = pd.DataFrame({L: 100 * R2[LEG == L].mean(0) for L in LEGS}, index=TEMAS).round(1)
heat = heat.loc[heat.mean(1).sort_values(ascending=False).index]
heat.to_csv(SAIDA / "figura_5_2_R2_por_legislatura.csv", encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(10, 6.7))
sns.heatmap(heat, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={"label": "% de discursos (regra R2)"}, ax=ax)
ax.set_title("Prevalência de temas por legislatura (regra R2)")
ax.set_xlabel("Legislatura"); ax.set_ylabel("")
plt.tight_layout()
fig.savefig(SAIDA / "figura2_agenda_por_legislatura.png", dpi=300)
plt.show()
heat

In [ ]:
# 9) TABELA 5.14 — controle do tamanho do discurso (regressão logística, como no notebook 18)
import warnings; warnings.filterwarnings("ignore")
import statsmodels.formula.api as smf
base = pd.DataFrame({"leg": df["_leg"].astype(str), "log_tokens": np.log(df["_n"].clip(lower=1)), "_leg": df["_leg"]})
linhas = []
for j, tema in enumerate(TEMAS):
    d = base.assign(R2=R2[:, j].astype(int))
    m = smf.logit("R2 ~ C(leg, Treatment('52')) + log_tokens", data=d).fit(disp=0)
    or_tam = float(np.exp(m.params["log_tokens"]))
    for L in LEGS:
        ajust = float(m.predict(d.assign(leg=str(L))).mean()) * 100
        bruto = float(d.loc[d["_leg"] == L, "R2"].mean()) * 100
        linhas.append({"tema": tema, "legislatura": L, "R2_bruto_%": round(bruto, 1),
                       "R2_ajustado_%": round(ajust, 1), "OR_log_tokens": round(or_tam, 2)})
    print(f"{tema}: pronto (razão de chances do log do tamanho = {or_tam:.2f})")
res = pd.DataFrame(linhas)
p = res.pivot(index="tema", columns="legislatura", values=["R2_bruto_%", "R2_ajustado_%"])
t514 = pd.DataFrame({"bruto_52": p[("R2_bruto_%", 52)], "bruto_57": p[("R2_bruto_%", 57)],
                     "ajust_52": p[("R2_ajustado_%", 52)], "ajust_57": p[("R2_ajustado_%", 57)]})
t514["var_bruta_pp"] = (t514["bruto_57"] - t514["bruto_52"]).round(1)
t514["var_ajustada_pp"] = (t514["ajust_57"] - t514["ajust_52"]).round(1)
t514 = t514.sort_values("var_bruta_pp")
res.to_csv(SAIDA / "tabela_5_14_completa.csv", index=False, encoding="utf-8-sig")
t514.to_csv(SAIDA / "tabela_5_14_resumo.csv", encoding="utf-8-sig")
print("Razões de chances do log do tamanho: entre", res["OR_log_tokens"].min(), "e", res["OR_log_tokens"].max())
t514

In [ ]:
# 10) TABELA A.1 — dez termos de maior frequência documental por categoria
linhas = []
for tema in TEMAS:
    top = sorted(THEMES[tema], key=lambda t: -docfreq[t])[:10]
    for t in top:
        linhas.append({"Categoria": tema, "Termo": t, "Discursos com o termo": docfreq[t]})
tA1 = pd.DataFrame(linhas)
tA1.to_csv(SAIDA / "tabela_A_1_termos_mais_frequentes.csv", index=False, encoding="utf-8-sig")
del dist, occ, dens
tA1

In [ ]:
# 11) CAMADA 2 — sinalização por discurso (dicionário setorizado final)
from bisect import bisect_left
def r2(cont, dic):
    for termos in dic.values():
        hits = [t for t in termos if cont[t] > 0]
        if len(hits) >= 2 or sum(cont[t] for t in hits) >= 3:
            return True
    return False
def perto(pos_a, pos_b, k):
    if not pos_a or not pos_b:
        return False
    for p_ in pos_a:
        i = bisect_left(pos_b, p_)
        if i < len(pos_b) and pos_b[i] - p_ <= k:
            return True
        if i > 0 and p_ - pos_b[i - 1] <= k:
            return True
    return False

TIPOS = list(SETORIZADO)
nacA = np.zeros(N, bool); nacB = np.zeros(N, bool); nacC = np.zeros(N, bool)
set1 = np.zeros(N, bool); nac2 = np.zeros(N, bool); set2 = np.zeros(N, bool)
j20 = np.zeros(N, bool); j50 = np.zeros(N, bool)
tipo = np.zeros((N, len(TIPOS)), bool)
for i, toks in enumerate(df["_tokens"]):
    pn = [k for k, t in enumerate(toks) if t in TN["A"]]
    ps = [k for k, t in enumerate(toks) if t in TERMOS_SET]
    nacA[i], set1[i] = bool(pn), bool(ps)
    st = set(toks)
    nacB[i] = not st.isdisjoint(TN["B"])
    nacC[i] = not st.isdisjoint(TN["C"])
    if ps:
        for j, tp in enumerate(TIPOS):
            tipo[i, j] = not st.isdisjoint(SETORIZADO[tp])
    if pn or ps:
        cont = Counter(toks)
        nac2[i] = r2(cont, NAC_A)
        set2[i] = r2(cont, SETORIZADO)
    if pn and ps:
        j50[i] = perto(pn, ps, 50)
        if j50[i]:
            j20[i] = perto(pn, ps, 20)
    if i % 50_000 == 0:
        print(f"  {i:,}/{N:,}")
mis = nacA & set1
print("Pronto.")
print(f"Agenda nacional: {100*nacA.mean():.1f}%")
print(f"Agenda setorizada: {100*set1.mean():.1f}%")
print(f"Mistos: {100*mis.mean():.1f}%")
print(f"Setorizada entre os com agenda nacional: {100*mis.sum()/nacA.sum():.1f}%")
print(f"Setorizada entre os SEM agenda nacional: {100*set1[~nacA].mean():.1f}%")

In [ ]:
# 12) TABELA 5.4 e FIGURA 5.3 — estrutura por legislatura
t54 = pd.DataFrame({
    "Agenda nacional (%)": [100 * nacA[LEG == L].mean() for L in LEGS],
    "Agenda setorizada (%)": [100 * set1[LEG == L].mean() for L in LEGS],
    "Mista (%)": [100 * mis[LEG == L].mean() for L in LEGS],
    "Setorizada entre nacional (%)": [100 * mis[LEG == L].sum() / nacA[LEG == L].sum() for L in LEGS],
}, index=LEGS).round(1)
t54.index.name = "Legislatura"
t54.to_csv(SAIDA / "tabela_5_4_estrutura_por_legislatura.csv", encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(10, 6.2))
ax.plot(LEGS, t54["Agenda nacional (%)"], marker="o", lw=2, label="Agenda nacional")
ax.plot(LEGS, t54["Agenda setorizada (%)"], marker="s", lw=2, label="Agenda setorizada")
ax.plot(LEGS, t54["Mista (%)"], marker="^", lw=2, label="Mista (nacional + setorizada)")
lo = 5 * np.floor(t54[["Agenda nacional (%)", "Agenda setorizada (%)", "Mista (%)"]].values.min() / 5) - 5
ax.set_ylim(lo, 95)
ax.set_title("Estrutura da agenda nacional e setorizada por legislatura")
ax.set_xlabel("Legislatura"); ax.set_ylabel("% de discursos"); ax.set_xticks(LEGS)
ax.grid(alpha=0.3); ax.legend(loc="upper right")
plt.tight_layout()
fig.savefig(SAIDA / "figura3_agenda_nacional_setorizada.png", dpi=300)
plt.show()
t54

In [ ]:
# 13) TABELA 5.5 — prevalência dos tipos de agenda setorizada
nomes = {"territorio_regiao": "Território e região", "raca_indigenas_minorias": "Raça, indígenas e minorias",
         "religiao_moral": "Religião e moral", "mulheres_familia_infancia": "Mulheres, família e infância",
         "trabalhadores_sindicatos": "Trabalhadores e sindicatos", "empresas_industria": "Empresas e indústria",
         "servidores_profissoes": "Servidores públicos e profissões", "agricultura_rural": "Agricultura e meio rural",
         "forcas_seguranca": "Forças de segurança"}
t55 = pd.DataFrame({"% de discursos": (100 * tipo.mean(0)).round(1)}, index=[nomes[t] for t in TIPOS])
t55 = t55.sort_values("% de discursos", ascending=False)
t55.index.name = "Tipo de setorização"
t55.to_csv(SAIDA / "tabela_5_5_tipos_setorizada.csv", encoding="utf-8-sig")
t55

In [ ]:
# 14) TABELA 5.15 — setorizada entre nacionais por regra (R1, R2, janelas)
regras = {"R1 (principal)": (nacA, mis), "R2": (nac2, nac2 & set2),
          "Janela de 50 tokens": (nacA, j50), "Janela de 20 tokens": (nacA, j20)}
t515 = pd.DataFrame({nome: [100 * m[LEG == L].sum() / n[LEG == L].sum() for L in LEGS] + [100 * m.sum() / n.sum()]
                     for nome, (n, m) in regras.items()}, index=LEGS + ["Total"]).T.round(1)
t515.to_csv(SAIDA / "tabela_5_15_regras_e_janelas.csv", encoding="utf-8-sig")
t515

In [ ]:
# 15) TABELA 5.16 — três versões do dicionário de agenda nacional
linhas = []
for v, nome in [("A", "Principal (Camada 2)"), ("B", "Sem termos sobrepostos"), ("C", "Dicionário da Camada 1 ajustado")]:
    nac = {"A": nacA, "B": nacB, "C": nacC}[v]
    m = nac & set1
    por_leg = [100 * m[LEG == L].mean() for L in LEGS]
    linhas.append({"Versão do dicionário": nome, "Agenda nacional (%)": round(100 * nac.mean(), 1),
                   "Setorizada entre nacionais (%)": round(100 * m.sum() / nac.sum(), 1),
                   "Mistos por legislatura (%)": f"{min(por_leg):.1f}–{max(por_leg):.1f}"})
t516 = pd.DataFrame(linhas).set_index("Versão do dicionário")
t516.to_csv(SAIDA / "tabela_5_16_versoes_dicionario.csv", encoding="utf-8-sig")
t516

In [ ]:
# 16) Razões de chances da Camada 2 (como no notebook 22)
dd = pd.DataFrame({"s": set1.astype(int), "n": nacA.astype(int), "lt": np.log(df["_n"].clip(lower=1)), "leg": df["_leg"]})
linhas = []
for nome, f in [("sem controle", "s ~ n"), ("com controle de comprimento", "s ~ n + lt"),
                ("com comprimento e legislatura", "s ~ n + lt + C(leg)")]:
    m = smf.logit(f, dd).fit(disp=0)
    ic = np.exp(m.conf_int().loc["n"]).round(2).tolist()
    linhas.append({"modelo": nome, "razao_de_chances": round(float(np.exp(m.params["n"])), 2),
                   "IC95_inf": ic[0], "IC95_sup": ic[1]})
rc = pd.DataFrame(linhas).set_index("modelo")
rc.to_csv(SAIDA / "razao_de_chances_camada2.csv", encoding="utf-8-sig")
print("Antes: 4,53 | 1,57 | 1,54")
rc

In [ ]:
# 17) Resumo dos principais indicadores
print("=" * 60)
print(f"Limiar fixo da R3: {LIMIAR:.2f}")
print(f"Agenda nacional: {100*nacA.mean():.1f}%")
print(f"Agenda setorizada: {100*set1.mean():.1f}%")
print(f"Mistos: {100*mis.mean():.1f}%")
print(f"Setorizada entre os com agenda nacional: {100*mis.sum()/nacA.sum():.1f}%")
print(f"Setorizada entre os sem agenda nacional: {100*set1[~nacA].mean():.1f}%")
print(f"Mistos por legislatura: {t54['Mista (%)'].min():.1f} a {t54['Mista (%)'].max():.1f}")
print("Razões de chances:", " | ".join(str(x) for x in rc["razao_de_chances"]), "")
seg = t514.loc["Segurança pública e violência"]
eco = t514.loc["Economia, trabalho e renda"]
print(f"Segurança 52->57: {seg['bruto_52']} -> {seg['bruto_57']} (bruto {seg['var_bruta_pp']:+} p.p.; ajustado {seg['var_ajustada_pp']:+} p.p.)")
print(f"Economia 52->57: {eco['bruto_52']} -> {eco['bruto_57']} (bruto {eco['var_bruta_pp']:+} p.p.; ajustado {eco['var_ajustada_pp']:+} p.p.)")
print("=" * 60)
print("Arquivos salvos em:", SAIDA)
for f in sorted(SAIDA.iterdir()):
    print("  ", f.name)